# Modelling: LightGBM and three other algorithms on the test shop

This notebook trains forecasting models on the feature frame built by [preprocessing.ipynb](preprocessing.ipynb), compares them in a **rolling-window backtest** with the competition metric, and writes a submission. The strategy follows [DATASETS.md](DATASETS.md) §8.

**Before running:** run `preprocessing.ipynb` once. Its last cell saves `processed/features_wide.parquet`, which this notebook loads.

**Models**

| Model | Family | Trained on | Loss / rule | Why it is here |
|---|---|---|---|---|
| `median_13w` | reference | each article's own history | median of the last 13 weeks | the baseline to beat (DATASETS §8.1) |
| `naive_last`, `seasonal_naive_52` | reference | each article's own history | last week / same weeks last year | the site's simple baselines, to relate offline scores to the site's (§9) |
| `lightgbm` | gradient-boosted trees, **global** | all series (test shop + A + C + D) | L1 on the scaled target | the main model (§8.4). L1 = absolute error, which is what MASE measures |
| `linear_l1` | linear, **global** | same | L1 (least absolute deviation, SGD) | shows how much the trees' non-linearity adds |
| `mlp` | neural network, **global** | same | squared error on `log1p(scaled target)` | a different learner for the ensemble |
| `tsb` | Croston family, **local** | each article's own history | TSB smoothing, median rule | the intermittent-demand specialist (§8.6) |

All global models predict `target / scale`, where `scale` is the series' mean absolute week-to-week change up to the origin. The loss is then close to MASE itself, and a 494-unit week in Set A doesn't dominate. Predictions are multiplied back by `scale`.

**How it is validated:** a rolling backtest split by time (no random split, §7), honest **test windows** that were never used for tuning, an overfitting check (§8), a comparison with the site's AutoARIMA / naive scores (§9) and a rounding test (§10). §14 ends with a bar chart of all scores.

**What to edit:** the configuration cell in §6 (`MODELS`, `ENSEMBLES`, `POSTPROCESS`, `N_WINDOWS`, `ORIGIN_STRIDE`). Every model is a function `fit_x(ctx, **options) -> (forecast matrix, fitted model)`, so adding one means writing one function and adding one line to `MODELS`.

**Runtime:** about 20–25 minutes on 16 cores with the defaults (4 windows + the final fit). LightGBM and the MLP take most of it. Use `N_WINDOWS = 1` or larger `ORIGIN_STRIDE` values for quick experiments.

In [ ]:
%pip install -q numpy pandas matplotlib pyarrow scikit-learn lightgbm

In [ ]:
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Works whether the kernel starts in the notebook folder or the project root
BASE = Path('.') if Path('set-a').exists() else Path('data/hackathon-retail-sets')
FEATURES_PATH = BASE / 'processed' / 'features_wide.parquet'
SUBMISSION_DIR = BASE / 'submissions'

H = 13
SID = 'series_id'
SEED = 0

# Plot style: same as the EDA notebooks
BLUE, GREY, INK = '#2a78d6', '#b5b4ae', '#52514e'
plt.rcParams.update({
    'figure.figsize': (11, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GREY, 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})

## 1. Load the feature frame and the raw test-shop history

- `frame`: one row per series × origin week with features and `target_h1 … target_h13` (preprocessing §8).
- `Y`: the raw test-shop matrix (articles × weeks). Scores are always computed on it, never on the feature frame.

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'{FEATURES_PATH} not found: run preprocessing.ipynb first (its §11 saves it).')

frame = pd.read_parquet(FEATURES_PATH)
print(f'feature frame: {len(frame):,} rows x {frame.shape[1]} cols | series per source:',
      frame.groupby('source', observed=True)[SID].nunique().to_dict())

ts = pd.read_parquet(BASE / 'test-shop' / 'history.parquet')
ts['week_start'] = pd.to_datetime(ts['week_start'])
Y = ts.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)
print(f'test shop: {Y.shape[0]:,} articles x {Y.shape[1]} weeks ({Y.columns[0].date()} -> {Y.columns[-1].date()})')

## 2. Helpers (same as preprocessing.ipynb §9–10)

Copied here so this notebook runs on its own.

- `split_frame` / `subsample_origins` / `to_long` build the train / validation / prediction rows for a cutoff.
- `mase_score` is the competition metric (from the EDA notebook): w1–w13, the live w1–w7 and the final w8–w13.

In [ ]:
META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'store_id', 'article_id', 'scale', 'is_outlier',
             'sample_weight', 'supplier_id', 'product_group', 'division', 'department', 'department_group',
             'store_format', 'category', 'subcategory', 'segment']
LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'sample_weight']


def rolling_cutoffs(frame, n_windows=4, step=H, horizon=H, source='test_shop'):
    last = frame.loc[frame['source'] == source, 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def split_frame(frame, cutoff=None, horizon=H, valid_source='test_shop'):
    is_v = frame['source'] == valid_source
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    target_end = frame['week_start'] + pd.Timedelta(weeks=horizon)
    train = frame[~is_v | (target_end <= cutoff)]
    valid = frame[is_v & (frame['week_start'] == cutoff)]
    return train.reset_index(drop=True), valid.reset_index(drop=True)


def subsample_origins(frame, every=None, min_history_weeks=1):
    keep = pd.Series(True, index=frame.index)
    if every:
        k = frame['source'].astype(str).map(every).fillna(1).astype(int)
        from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
        keep &= (from_end % k == 0)
    if 'age_weeks' in frame:
        keep &= frame['age_weeks'] >= min_history_weeks - 1
    return frame[keep].reset_index(drop=True)


def to_long(frame, horizon=H, drop_missing_target=True, id_cols=LONG_ID_COLS):
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base = [c for c in id_cols if c in frame.columns] + feats
    parts = []
    for h in range(1, horizon + 1):
        p = frame[base].copy()
        p['h'] = np.int8(h)
        if f'seasonal_h{h}' in frame:
            p['seasonal_lag52'] = frame[f'seasonal_h{h}'].to_numpy()
        if f'target_h{h}' in frame:
            p['target'] = frame[f'target_h{h}'].to_numpy()
        parts.append(p)
    long = pd.concat(parts, ignore_index=True)
    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target and 'target' in long:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long


def forecast_matrix(rows, pred, index=Y.index, horizon=H):
    """Long rows (article_id, h) + predictions -> articles x horizon array; missing articles -> 0."""
    F = pd.DataFrame({'article_id': rows['article_id'].astype(str).to_numpy(), 'h': rows['h'].to_numpy(),
                      'pred': np.asarray(pred, dtype=float)}).pivot(index='article_id', columns='h', values='pred')
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


def per_article_mase(cutoff, F, Y=Y, horizon=H):
    """MASE per article (NaN for articles with no movement before the cutoff, which can't be scored)."""
    hist = Y.loc[:, :cutoff]
    act = Y.loc[:, Y.columns > cutoff].iloc[:, :horizon].to_numpy()
    den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
    err = np.abs(act - np.clip(F, 0, None))
    with np.errstate(divide='ignore', invalid='ignore'):
        return {name: np.where(den > 0, err[:, cols].mean(axis=1) / den, np.nan)
                for name, cols in [('all', slice(0, horizon)), ('w1-w7', slice(0, 7)), ('w8-w13', slice(7, horizon))]}


def mase_score(cutoff, F):
    """Competition score (1 - MASE) x 100, higher is better."""
    return {k: round((1 - np.nanmean(v)) * 100, 2) for k, v in per_article_mase(cutoff, F).items()}


def make_submission(F, index=Y.index):
    F = np.clip(np.asarray(F, dtype=float), 0, None)
    sub = pd.DataFrame(F, index=index, columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == 2904 and sub.article_id.is_unique and not sub.isna().any().any()
    return sub

## 3. Backtest context

`make_context` builds everything a model needs for one cutoff, once, so all models see exactly the same data:

| key | content |
|---|---|
| `cutoff` | last known week (`None` for the final fit = the last week of history) |
| `train` | long training rows: every origin of the extra sets, and test-shop origins whose 13 target weeks end by the cutoff (skipped with `with_train=False`) |
| `valid` | long rows of the test shop at the cutoff (all 2,904 articles × 13 horizons) |
| `features` | model input columns |
| `Y_hist` | raw test-shop matrix up to the cutoff, for the local models |
| `weeks_since_sale`, `demand_class` | per article at the cutoff, for post-processing and the error breakdown |

In [ ]:
CAT_FEATURES = ['demand_class']


def make_context(frame, cutoff=None, origin_stride=None, with_train=True):
    train_w, valid_w = split_frame(frame, cutoff)
    cutoff = valid_w['week_start'].max()
    valid = to_long(valid_w, drop_missing_target=False)
    at_cut = valid_w.assign(article_id=valid_w['article_id'].astype(str)).set_index('article_id').reindex(Y.index)
    return {
        'cutoff': cutoff,
        'train': to_long(subsample_origins(train_w, every=origin_stride)) if with_train else None,
        'valid': valid,
        'features': [c for c in valid.columns if c not in LONG_ID_COLS and c != 'target'],
        'Y_hist': Y.loc[:, :cutoff],
        'weeks_since_sale': at_cut['weeks_since_sale'].to_numpy(float),
        'demand_class': at_cut['demand_class'].astype(str).to_numpy(),
    }

## 4. Models

Each `fit_*` function trains on `ctx['train']` (or reads `ctx['Y_hist']` for local models) and returns `(F, fitted)`:
- `F`: an articles × 13 array of forecasts in raw units, in the order of `Y.index`, for the context's cutoff.
- `fitted.predict(other_ctx)`: the same fitted model applied to another cutoff. §8 uses this to score each model on weeks it was trained on, to check for overfitting. `fitted.model` is the underlying object (e.g. the LightGBM booster).

**Reference models** (no training): `median_13w` is the baseline to beat (DATASETS §8.1). `naive_last` (last week's value) and `seasonal_naive_52` (the same weeks last year) are the site's simple baselines, used in §9 to relate our offline scores to the site's AutoARIMA and naive scores.

**Notes on the four learners**
- **LightGBM**: `objective='l1'` because MASE is an absolute error. `demand_class` is a native categorical. Rows are weighted with `sample_weight` (test shop ×3, set in preprocessing).
- **Linear L1**: `SGDRegressor` with the epsilon-insensitive loss and `epsilon=0`, which is exactly the absolute loss. It needs clean inputs, so features go through `signed_log → median imputation (+ missing flags) → standard scaling`, and `demand_class` / `h` are one-hot encoded. It trains on a random sample of rows.
- **MLP**: scikit-learn's MLP only has squared loss. Training on `log1p(target)` turns that into something closer to a median for skewed demand. The log model still predicts small positive values for series that are mostly zero, and the median of such a week is 0, so predictions below `zero_below` (in scaled units) are set to 0.
- **TSB** (Teunter–Syntetos–Babai) smooths two things per article: the probability of a sale `p` (updated every week, so it decays when an article stops selling) and the size of a sale `z` (updated on sale weeks). Classic TSB forecasts the mean `p·z`. Under absolute error the best forecast is the *median*: `z` if `p > threshold` (0.5 = the textbook median), otherwise 0. That is `mode='median'`, which scored about 20 points better in testing.

In [ ]:
class Fitted:
    """A fitted model: .predict(ctx) -> forecast matrix for ctx's cutoff; .model = underlying object."""
    def __init__(self, predict, model=None):
        self.predict, self.model = predict, model


def _fitted(ctx, predict, model=None):
    return predict(ctx), Fitted(predict, model)


def _repeat(f):
    return np.repeat(np.asarray(f, dtype=float)[:, None], H, axis=1)


# ---------- reference models ----------
def fit_median(ctx, weeks=13):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -weeks:].median(axis=1)))


def fit_naive(ctx):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -1]))


def fit_seasonal_naive(ctx, season=52):
    return _fitted(ctx, lambda c: c['Y_hist'].iloc[:, -season:-season + H].to_numpy(float))


# ---------- LightGBM ----------
LGB_PARAMS = dict(objective='l1', learning_rate=0.1, num_leaves=63, min_data_in_leaf=1000, max_bin=63,
                  feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=1.0,
                  force_col_wise=True, verbose=-1, seed=SEED)


def _global_predictor(model, feats, transform=None):
    """predict(ctx): model output on ctx['valid'] -> (optional transform) -> x scale -> forecast matrix."""
    def predict(c):
        va = c['valid']
        out = model.predict(va[feats])
        if transform is not None:
            out = transform(out)
        return forecast_matrix(va, np.clip(out, 0, None) * va['scale'].to_numpy())
    return predict


def fit_lightgbm(ctx, params=LGB_PARAMS, rounds=200, use_weights=True):
    tr, feats = ctx['train'], ctx['features']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'] if use_weights else None,
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- shared input pipeline for the linear model and the MLP ----------
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_tabular_preprocessor(features, categorical=(*CAT_FEATURES, 'h')):
    cat = [c for c in categorical if c in features]
    num = [c for c in features if c not in cat]
    return ColumnTransformer([
        ('num', make_pipeline(FunctionTransformer(signed_log),
                              SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])


def _sample(df, n, seed=SEED):
    return df.sample(n=n, random_state=seed) if n and len(df) > n else df


# ---------- linear L1 ----------
def fit_linear_l1(ctx, sample=1_000_000, alpha=1e-5, epochs=10, target_clip_q=0.999, use_weights=True):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, alpha=alpha, average=True,
                                       max_iter=epochs, tol=None, random_state=SEED))
    y = tr['target'].clip(upper=tr['target'].quantile(target_clip_q))   # scaled targets can explode for tiny scales
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], y, sgdregressor__sample_weight=tr['sample_weight'].to_numpy() if use_weights else None)
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- MLP ----------
def fit_mlp(ctx, sample=500_000, hidden=(128, 64), max_iter=30, alpha=1e-4, zero_below=0.5):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          MLPRegressor(hidden_layer_sizes=hidden, batch_size=2048, learning_rate_init=1e-3,
                                       alpha=alpha, max_iter=max_iter, early_stopping=True, n_iter_no_change=4,
                                       random_state=SEED))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], np.log1p(tr['target'].clip(lower=0)))

    def back(out):
        out = np.expm1(out).clip(0)
        return np.where(out < zero_below, 0, out)
    return _fitted(ctx, _global_predictor(model, feats, back), model)


# ---------- TSB ----------
def tsb_state(Y_hist, alpha=0.1, beta=0.05):
    """Run TSB over the history, vectorised across articles. Returns (p, z) at the last week."""
    X = Y_hist.to_numpy(float)
    p, z = np.zeros(len(X)), np.full(len(X), np.nan)
    for t in range(X.shape[1]):
        d = X[:, t]
        sale = d > 0
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = p + beta * (sale - p)
    return p, np.nan_to_num(z)


def fit_tsb(ctx, alpha=0.1, beta=0.05, mode='median', threshold=0.5):
    def predict(c):
        p, z = tsb_state(c['Y_hist'], alpha, beta)
        return _repeat(p * z if mode == 'mean' else np.where(p > threshold, z, 0.0))
    return _fitted(ctx, predict)

## 5. Post-processing

Applied to every model's forecast after `clip ≥ 0`. Both raw and post-processed scores are reported, so you can see what each step does.

- `dead_to_zero`: forecast 0 for articles with no sale in the last `weeks` weeks (DATASETS §8.1; 1,073 articles at the end of history).
- `snap_small`: forecasts below `below` units → 0. Over-forecasting slow sellers is the biggest MASE risk (§2), and the median of a mostly-zero week is 0.
- `round_to_int`: round to whole units. Off by default; §10 tests whether it helps.

In [ ]:
def dead_to_zero(F, ctx, weeks=8):
    dead = ~(ctx['weeks_since_sale'] < weeks)          # NaN (never sold) counts as dead
    return np.where(dead[:, None], 0.0, F)


def snap_small(F, ctx, below=0.5):
    return np.where(F < below, 0.0, F)


def round_to_int(F, ctx):
    return np.round(F)


def postprocess(F, ctx, steps):
    F = np.clip(F, 0, None)
    for step in steps:
        func, kw = (step, {}) if callable(step) else step
        F = func(F, ctx, **kw)
    return F

## 6. Configure ← edit here

- `MODELS`: name → `(fit function, options)`. Comment out a line to skip a model.
- `REFERENCES`: models that are only there for comparison. They are never chosen as the final model.
- `ENSEMBLES`: name → `(combine function, [model names])`. Built from the members' raw forecasts and post-processed like a model.
- `POSTPROCESS`: steps from §5, in order.
- `N_WINDOWS`: number of 13-week backtest windows (DATASETS §8.2 recommends several: scores move ~25 points between windows).
- `TUNING_WINDOWS`: how many of the most recent windows were used to tune settings and pick the final model. The rest are the **test windows** (see §7).
- `ORIGIN_STRIDE`: keep every k-th origin per source in the training rows. Smaller = more data, slower.

In [ ]:
MODELS = {
    'median_13w':        (fit_median, {'weeks': 13}),
    'naive_last':        (fit_naive, {}),
    'seasonal_naive_52': (fit_seasonal_naive, {}),
    'lightgbm':          (fit_lightgbm, {'rounds': 200}),
    'linear_l1':         (fit_linear_l1, {'sample': 1_000_000}),
    'mlp':               (fit_mlp, {'sample': 500_000, 'hidden': (128, 64)}),
    'tsb':               (fit_tsb, {'alpha': 0.1, 'beta': 0.05, 'mode': 'median'}),
}
REFERENCES = ['median_13w', 'naive_last', 'seasonal_naive_52']
BASELINE = 'median_13w'

ENSEMBLES = {
    'ens_mean(lgb+linear)': (np.mean, ['lightgbm', 'linear_l1']),
    'ens_median(4 models)': (np.median, ['lightgbm', 'linear_l1', 'mlp', 'tsb']),
}

POSTPROCESS = [
    (dead_to_zero, {'weeks': 8}),
    (snap_small, {'below': 0.5}),
]

N_WINDOWS = 4
TUNING_WINDOWS = 1
ORIGIN_STRIDE = {'test_shop': 2, 'A': 4, 'C': 4, 'D': 2}

# Scores published by the competition site, measured on the real hidden weeks (live leaderboard = w1-w7)
SITE_SCORES = {'AutoARIMA (site)': -33.7, 'naive (site)': -64.5}

## 7. Rolling backtest (the train / validation / test design)

**No random split.** Shuffling weeks would let a model learn from the future. Every split here is by time:

- For a **cutoff**, models train only on weeks up to the cutoff (test-shop origins whose 13 target weeks end by the cutoff, plus the extra sets) and are scored on the **13 weeks after it**, exactly like the real task.
- This is repeated for `N_WINDOWS` cutoffs, 13 weeks apart.
- **Tuning vs test windows.** The model settings (LightGBM rounds and learning rate, the MLP's `zero_below`, the TSB median rule, the post-processing) were chosen by looking at the **most recent window only**, and `FINAL_MODEL` is picked there too. So that window is the *validation / tuning* window, and its score is slightly optimistic. The **older windows were never used for any decision**, which makes them the honest **test windows**. (The usual order is reversed because the tuning had already happened on the latest window; what matters is that the test windows stay untouched.)
- **In-sample check.** Each fitted model is also scored on the 13 weeks *before* its cutoff. Those weeks were training targets for the global models, so comparing the two scores shows overfitting (§8).

All forecasts are kept in `forecasts[(model, cutoff)]` (out-of-sample) and `forecasts_in[(model, cutoff)]` (in-sample). The fitted models of the most recent window are kept in `fitted`.

In [ ]:
def combine(store, members, how, cutoff):
    return how(np.stack([store[(m, cutoff)] for m in members]), axis=0)


def _score_rows(name, cutoff, F, F_in, ctx, ctx_in, post_steps, secs):
    rows = []
    for variant, G, c in [('raw', np.clip(F, 0, None), ctx), ('post', postprocess(F, ctx, post_steps), ctx),
                          ('in-sample', postprocess(F_in, ctx_in, post_steps), ctx_in)]:
        rows.append({'model': name, 'cutoff': cutoff.date(), 'variant': variant, 'fit_s': round(secs, 1),
                     **mase_score(c['cutoff'], G)})
    return rows


def backtest(frame, models, ensembles, cutoffs, post_steps, origin_stride):
    rows, forecasts, forecasts_in, fitted, contexts = [], {}, {}, {}, {}
    for i, cutoff in enumerate(cutoffs):
        t0 = time.perf_counter()
        ctx = make_context(frame, cutoff, origin_stride)
        ctx_in = make_context(frame, cutoff - pd.Timedelta(weeks=H), with_train=False)   # weeks inside training
        contexts[cutoff] = {k: v for k, v in ctx.items() if k not in ('train', 'valid')}
        print(f'\n=== cutoff {cutoff.date()} | train {len(ctx["train"]):,} rows | context {time.perf_counter() - t0:.0f}s')
        print(f'  {"model":<22s} {"in-sample":>9s} {"raw":>8s} {"post":>8s}')
        for name, (fit, kw) in models.items():
            t0 = time.perf_counter()
            F, model = fit(ctx, **kw)
            secs = time.perf_counter() - t0
            forecasts[(name, cutoff)], forecasts_in[(name, cutoff)] = F, model.predict(ctx_in)
            if i == 0:
                fitted[name] = model
            new = _score_rows(name, cutoff, F, forecasts_in[(name, cutoff)], ctx, ctx_in, post_steps, secs)
            rows += new
            print(f'  {name:<22s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}   ({secs:.0f}s)')
        for name, (how, members) in ensembles.items():
            if all((m, cutoff) in forecasts for m in members):
                forecasts[(name, cutoff)] = combine(forecasts, members, how, cutoff)
                forecasts_in[(name, cutoff)] = combine(forecasts_in, members, how, cutoff)
                new = _score_rows(name, cutoff, forecasts[(name, cutoff)], forecasts_in[(name, cutoff)],
                                  ctx, ctx_in, post_steps, 0.0)
                rows += new
                print(f'  {name:<22s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}')
        del ctx
    return pd.DataFrame(rows), forecasts, forecasts_in, fitted, contexts


cutoffs = rolling_cutoffs(frame, n_windows=N_WINDOWS)
tuning_cutoffs, test_cutoffs = cutoffs[:TUNING_WINDOWS], cutoffs[TUNING_WINDOWS:]
print('Tuning window(s):', [c.date() for c in tuning_cutoffs], '| test windows:', [c.date() for c in test_cutoffs])
results, forecasts, forecasts_in, fitted, contexts = backtest(frame, MODELS, ENSEMBLES, cutoffs, POSTPROCESS, ORIGIN_STRIDE)

### Results: tuning window vs test windows

Score = (1 − MASE) × 100, **higher is better** (0 = off by one typical weekly move, 100 = perfect). **`w8-w13` is what the final ranking uses** (DATASETS §2). All scores are post-processed.

In [ ]:
post = results[results.variant == 'post']


def window_mean(df, cuts, cols=('all', 'w1-w7', 'w8-w13')):
    return df[df.cutoff.isin([c.date() for c in cuts])].groupby('model')[list(cols)].mean()


summary = pd.concat({'tuning window': window_mean(post, tuning_cutoffs),
                     'TEST windows (untouched)': window_mean(post, test_cutoffs),
                     'all windows': window_mean(post, cutoffs)}, axis=1).round(2)
summary[('', 'beats baseline on every test window')] = [
    bool((post[(post.model == m) & post.cutoff.isin([c.date() for c in test_cutoffs])].set_index('cutoff')['all']
          > post[(post.model == BASELINE) & post.cutoff.isin([c.date() for c in test_cutoffs])].set_index('cutoff')['all']).all())
    for m in summary.index]
summary = summary.sort_values(('TEST windows (untouched)', 'all'), ascending=False)
display(summary)

print('Per window (all w1-w13):')
display(post.pivot(index='model', columns='cutoff', values='all').loc[summary.index])

## 8. Overfitting check

**Train vs unseen weeks.** For every model and window, compare the score on the 13 weeks *before* the cutoff (training targets for the global models) with the score on the 13 weeks *after* it (never seen). Some gap is expected even without overfitting, because the two periods differ in difficulty. The reference models have no training at all, so **their gap measures the period difference alone**. A learner overfits if its gap is clearly larger than the baseline's: that is the `excess gap` column.

In [ ]:
ins = results[results.variant == 'in-sample'].groupby('model')['all'].mean()
oos = post.groupby('model')['all'].mean()
gap = pd.DataFrame({'in-sample (trained-on weeks)': ins, 'out-of-sample (unseen weeks)': oos})
gap['gap'] = gap.iloc[:, 0] - gap.iloc[:, 1]
gap['excess gap vs baseline'] = gap['gap'] - gap.loc[BASELINE, 'gap']
display(gap.loc[summary.index].round(2))

**LightGBM learning curve.** Train one model with many boosting rounds on the tuning window and score it after each number of rounds, on both the trained-on weeks and the unseen weeks. If the unseen-weeks score peaks and then falls while the trained-on score keeps rising, extra rounds are overfitting. The dashed line marks the `rounds` used in `MODELS`.

In [ ]:
LC_ROUNDS = [25, 50, 100, 150, 200, 300, 400, 600]


def lgb_learning_curve(frame, cutoff, rounds_grid=LC_ROUNDS, params=LGB_PARAMS):
    ctx = make_context(frame, cutoff, ORIGIN_STRIDE)
    ctx_in = make_context(frame, cutoff - pd.Timedelta(weeks=H), with_train=False)
    feats, tr = ctx['features'], ctx['train']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'], categorical_feature=CAT_FEATURES)
    model = lgb.train(params, ds, num_boost_round=max(rounds_grid))
    rows = []
    for r in rounds_grid:
        row = {'rounds': r}
        for label, c in [('trained-on weeks', ctx_in), ('unseen weeks', ctx)]:
            va = c['valid']
            F = forecast_matrix(va, np.clip(model.predict(va[feats], num_iteration=r), 0, None) * va['scale'].to_numpy())
            row[label] = mase_score(c['cutoff'], postprocess(F, c, POSTPROCESS))['all']
        rows.append(row)
    return pd.DataFrame(rows).set_index('rounds')


if 'lightgbm' in MODELS:
    t0 = time.perf_counter()
    lc = lgb_learning_curve(frame, tuning_cutoffs[0])
    print(f'learning curve ({time.perf_counter() - t0:.0f}s)')
    display(lc)
    ORANGE = '#eb6834'
    fig, ax = plt.subplots(figsize=(9, 4))
    for col, color in [('trained-on weeks', ORANGE), ('unseen weeks', BLUE)]:
        ax.plot(lc.index, lc[col], color=color, linewidth=2, marker='o', markersize=5, label=col)
        ax.annotate(col, (lc.index[-1], lc[col].iloc[-1]), textcoords='offset points', xytext=(6, 0),
                    va='center', color=INK, fontsize=9)
    ax.axvline(MODELS['lightgbm'][1].get('rounds', 200), color=INK, linewidth=1, linestyle='--')
    ax.set_xlabel('boosting rounds'); ax.set_ylabel('score (higher is better)')
    ax.set_xlim(0, lc.index[-1] * 1.25)
    ax.legend(frameon=False, loc='lower right')
    ax.set_title(f'LightGBM score vs boosting rounds (window {tuning_cutoffs[0].date()})')
    plt.tight_layout(); plt.show()

## 9. How good is it, and does it beat the site's baselines?

The site's AutoARIMA (−33.7) and naive (−64.5) scores are measured on the **real hidden weeks**, which we can't score offline. Comparing them directly with our backtest scores would mix up two different periods. Instead, the site's own **naive** forecast serves as a calibration point: we run the same naive forecast in every backtest window and look for windows where it scores about the same as on the site. Those windows are about as hard as the real period.

AutoARIMA itself isn't in the backtest because it needs an extra package (`statsforecast`) and is slow on 2,904 series.

In [ ]:
cal = post.pivot(index='model', columns='cutoff', values='w1-w7').loc[summary.index]
print('Live-leaderboard weeks (w1-w7) per backtest window:')
display(cal.round(2))
nearest = (cal.loc['naive_last'] - SITE_SCORES['naive (site)']).abs().idxmin()
best = summary.index[~summary.index.isin(REFERENCES)][0]
print(f"Site naive on the real hidden w1-w7: {SITE_SCORES['naive (site)']} | closest backtest window: {nearest} "
      f"(our naive {cal.loc['naive_last', nearest]:.1f})")
print(f"In that window: {best} {cal.loc[best, nearest]:.1f}, median baseline {cal.loc[BASELINE, nearest]:.1f}, "
      f"vs site AutoARIMA {SITE_SCORES['AutoARIMA (site)']} on the real weeks.")

## 10. Should forecasts be rounded to whole units?

The competition accepts decimals, and MASE scores the absolute error of whatever is submitted. In theory, for whole-unit sales and absolute error, the best possible forecast is always a whole number (the median), but rounding a *given* forecast to the nearest integer isn't guaranteed to land on the better neighbour. Below, every model's post-processed forecast is scored with and without `round_to_int` on every window.

In [ ]:
rnd = []
for (m, c), F in forecasts.items():
    G = postprocess(F, contexts[c], POSTPROCESS)
    rnd.append({'model': m, 'cutoff': c.date(), 'decimals': mase_score(c, G)['all'],
                'rounded': mase_score(c, np.round(G))['all']})
rnd = pd.DataFrame(rnd)
rnd_t = rnd.groupby('model')[['decimals', 'rounded']].mean().round(2)
rnd_t['rounding changes score by'] = (rnd_t['rounded'] - rnd_t['decimals']).round(2)
rnd_t['rounding better on # windows'] = rnd.assign(b=rnd.rounded > rnd.decimals).groupby('model')['b'].sum()
display(rnd_t.loc[summary.index])

## 11. Where the error comes from

Mean MASE per demand class (at the cutoff) on the tuning window, post-processed. **Lower is better here.** DATASETS §8.6 suggests keeping whichever model wins per class. The `articles` row shows how many articles each class has, since every article weighs the same in the score.

In [ ]:
c0 = tuning_cutoffs[0]
ctx0 = contexts[c0]
by_class = {}
for m in summary.index:
    G = postprocess(forecasts[(m, c0)], ctx0, POSTPROCESS)
    by_class[m] = pd.Series(per_article_mase(c0, G)['all']).groupby(ctx0['demand_class']).mean()
cls = pd.DataFrame(by_class).T.round(3)
cls.loc['articles'] = pd.Series(ctx0['demand_class']).value_counts()
display(cls[[c for c in ['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales'] if c in cls]])

### TSB parameter sensitivity

TSB is cheap (milliseconds per fit), so its parameters can be scanned. To keep the test windows untouched, the scan uses the **tuning window only**; the test-window score of the top settings is shown for information, not for choosing.

In [ ]:
grid = []
for a in [0.05, 0.1, 0.2, 0.3]:
    for b in [0.02, 0.05, 0.1]:
        for thr in [0.4, 0.5, 0.6]:
            sc = {c: mase_score(c, postprocess(fit_tsb({'Y_hist': Y.loc[:, :c]}, a, b, 'median', thr)[0],
                                               contexts[c], POSTPROCESS))['all'] for c in cutoffs}
            grid.append({'alpha': a, 'beta': b, 'threshold': thr,
                         'tuning window': round(np.mean([sc[c] for c in tuning_cutoffs]), 2),
                         'test windows (info only)': round(np.mean([sc[c] for c in test_cutoffs]), 2) if test_cutoffs else np.nan})
display(pd.DataFrame(grid).sort_values('tuning window', ascending=False).head(10))

## 12. LightGBM feature importance

Gain importance from the model of the most recent window: how much each feature reduced the L1 loss in total.

In [ ]:
if 'lightgbm' in fitted:
    m = fitted['lightgbm'].model
    imp = pd.Series(m.feature_importance('gain'), index=m.feature_name()).sort_values()
    imp = (imp / imp.sum() * 100).tail(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('% of total gain')
    ax.set_title('LightGBM: top 20 features by gain (most recent window)')
    plt.tight_layout(); plt.show()

## 13. Final fit and submission

Refit on **all** history (cutoff = the last week, 2023-04-17), forecast 2023-04-24 → 2023-07-17 and write the CSV (`article_id, w1 … w13`).

`FINAL_MODEL` defaults to the best **tuning-window** score on w8–w13 (what the final ranking uses), among non-reference models. It is not chosen on the test windows, so their scores above stay an honest estimate. Set it by hand to override; it can be a model or an ensemble.

**Decimals:** the forecasts are kept as decimals unless `round_to_int` is added to `POSTPROCESS`; §10 shows whether that helps.

In [ ]:
candidates = [m for m in summary.index if m not in REFERENCES]
FINAL_MODEL = window_mean(post, tuning_cutoffs).loc[candidates, 'w8-w13'].idxmax()
SAVE = True

members = ENSEMBLES[FINAL_MODEL][1] if FINAL_MODEL in ENSEMBLES else [FINAL_MODEL]
print(f'Final model: {FINAL_MODEL} | tuning window w8-w13: {summary.loc[FINAL_MODEL, ("tuning window", "w8-w13")]} '
      f'| TEST windows all: {summary.loc[FINAL_MODEL, ("TEST windows (untouched)", "all")]} '
      f'(baseline {summary.loc[BASELINE, ("TEST windows (untouched)", "all")]})')

ctx_final = make_context(frame, cutoff=None, origin_stride=ORIGIN_STRIDE)
print(f'Training rows: {len(ctx_final["train"]):,} | forecast origin: {ctx_final["cutoff"].date()}')
final_fc = {}
for m in members:
    t0 = time.perf_counter()
    fit, kw = MODELS[m]
    final_fc[m] = fit(ctx_final, **kw)[0]
    print(f'  fitted {m} ({time.perf_counter() - t0:.0f}s)')

F_final = ENSEMBLES[FINAL_MODEL][0](np.stack([final_fc[m] for m in members]), axis=0) \
    if FINAL_MODEL in ENSEMBLES else final_fc[FINAL_MODEL]
F_final = postprocess(F_final, ctx_final, POSTPROCESS)
sub = make_submission(F_final)

print(f'Articles forecast 0 for all 13 weeks: {(F_final.sum(axis=1) == 0).sum():,} of {len(sub):,}')
print(f'Mean forecast per article-week: {F_final.mean():.2f} (last 13 weeks of history: {Y.iloc[:, -13:].to_numpy().mean():.2f})')
display(sub.head())

if SAVE:
    SUBMISSION_DIR.mkdir(exist_ok=True)
    path = SUBMISSION_DIR / f'submission_{re.sub(r"[^A-Za-z0-9]+", "_", FINAL_MODEL).strip("_")}.csv'
    sub.to_csv(path, index=False)
    print('Saved', path)

## 14. All model scores

Mean score on the **test windows** (never used for tuning), all 13 weeks, post-processed. Higher is better.

- **Blue**: our learners and ensembles.
- **Grey**: reference forecasts run in the same backtest.
- **Outlined**: the site's published scores. They are measured on the real hidden weeks (w1–w7), a different period from the backtest, so they show the scale rather than a like-for-like comparison (see §9).

In [ ]:
test_scores = summary[('TEST windows (untouched)', 'all')].rename(index=lambda m: f'{m} (final)' if m == FINAL_MODEL else m)
bars = pd.concat([test_scores, pd.Series(SITE_SCORES)]).sort_values()
kind = ['site' if m in SITE_SCORES else 'reference' if m in REFERENCES else 'ours' for m in bars.index]

fig, ax = plt.subplots(figsize=(10, 0.42 * len(bars) + 1.4))
for y, (m, v, k) in enumerate(zip(bars.index, bars.values, kind)):
    style = {'ours': dict(color=BLUE), 'reference': dict(color=GREY),
             'site': dict(facecolor='white', edgecolor=INK, linewidth=1.2)}[k]
    ax.barh(y, v, height=0.62, **style)
    ax.text(v - 0.8, y, f'{v:.1f}', va='center', ha='right', color=INK, fontsize=9)
ax.set_yticks(range(len(bars)), bars.index)
ax.axvline(0, color=INK, linewidth=0.8)
ax.set_xlim(bars.min() * 1.12, max(bars.max(), 0) + 2)
ax.grid(axis='y', visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='our models'), Patch(color=GREY, label='reference forecasts (same backtest)'),
                   Patch(facecolor='white', edgecolor=INK, label='site scores (real hidden weeks)')],
          frameon=False, loc='upper left')
ax.set_xlabel('score = (1 − MASE) × 100, higher is better')
ax.set_title('Model scores: mean of the test windows')
plt.tight_layout(); plt.show()